In [11]:
import numpy as np
from mpi4py import MPI

import dolfinx
import dolfinx.plot
import ufl
import basix.ufl
from dolfinx.fem.petsc import LinearProblem

from fenicsx_ii import assemble_scalar

from mesh import GenMesh3D

In [38]:
N = 16
comm = MPI.COMM_WORLD
_, _, graph_nodes, graph_cells = GenMesh3D(2*N)

ufl_int = ufl.Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(3,)))

graph_mesh = dolfinx.mesh.create_mesh(
    comm, 
    cells=graph_cells,
    e=ufl_int,
    x=graph_nodes
)

In [39]:
degree = 1
V = dolfinx.fem.functionspace(graph_mesh, ("CG", degree))
u, w = ufl.TrialFunction(V), ufl.TestFunction(V)

dx = ufl.Measure("dx", domain=graph_mesh)
dS = ufl.Measure("dS", domain=graph_mesh)
ds = ufl.Measure("ds", domain=graph_mesh)

n_graph = ufl.FacetNormal(graph_mesh)
h_graph = ufl.CellDiameter(graph_mesh)

In [40]:
# Bilinear Form
a = ufl.inner(ufl.grad(u), ufl.grad(w)) * dx

# Manufactured Solution
x = ufl.SpatialCoordinate(graph_mesh)

uhat_1 = x[2] + ufl.cos(2 * ufl.pi * x[2])
uhat_2 = 2 + 0.5 * ufl.sqrt(2) * (x[2] - 1)
u_ex = ufl.conditional(x[2] < 1, uhat_1, uhat_2)

# BC Expr
graph_mesh.topology.create_connectivity(graph_mesh.topology.dim - 1, graph_mesh.topology.dim)
exterior_facets = dolfinx.mesh.exterior_facet_indices(graph_mesh.topology)
exterior_dofs = dolfinx.fem.locate_dofs_topological(
    V, graph_mesh.topology.dim - 1, exterior_facets
)

bc_expr = dolfinx.fem.Expression(u_ex, V.element.interpolation_points)
u_bc = dolfinx.fem.Function(V)
u_bc.interpolate(bc_expr)
bc = dolfinx.fem.dirichletbc(u_bc, exterior_dofs)

# RHS Function
f = ufl.conditional(x[2] < 1, 4 * (ufl.pi ** 2) * ufl.cos(2 * ufl.pi * x[2]), 0)

# Linear Form
L = ufl.inner(f, w) * dx
#L += ufl.inner(u_bc * n_graph, ufl.grad(w)) * ds

In [41]:
problem = LinearProblem(
    a,
    L,
    bcs=[bc],
    petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
    petsc_options_prefix="Poisson",
)

uh = problem.solve()

In [42]:
print(uh.x.array)

[ 1.          0.98637953  0.83210678  0.57018344  0.25000001 -0.07018342
 -0.33210677 -0.48637951 -0.49999998 -0.36137951 -0.08210677  0.30481658
  0.75000001  1.19518344  1.58210678  1.86137953  2.          2.04419417
  2.04419417  2.08838835  2.08838835  2.13258252  2.13258252  2.1767767
  2.1767767   2.22097087  2.22097087  2.26516504  2.26516504  2.30935922
  2.30935922  2.35355339  2.35355339  2.39774756  2.39774756  2.44194174
  2.44194174  2.48613591  2.48613591  2.53033009  2.53033009  2.57452426
  2.57452426  2.61871843  2.61871843  2.66291261  2.66291261  2.70710678
  2.70710678]


In [43]:
def L2(f: ufl.core.expr.Expr, dx: ufl.Measure, domain: dolfinx.mesh.Mesh) -> float:
    integral = ufl.inner(f, f) * dx
    return np.sqrt(assemble_scalar(integral, op=MPI.SUM))

print(f"L2 Error: {L2(uh - u_ex, dx, graph_mesh)}")

L2 Error: 0.009920925613180314
